# Independent-area replication
## Question and why it matters
Does the excursion-cutoff association survive area matching from separate seeds?
See [registration](README.md) and [scientific context](../../../notes/open-questions-hypotheses.md).
## Assumptions and definitions
Seed modulo 3 defines three disjoint trajectory groups. At least two surviving
seeds per group are required. Independence between seeds is assumed, not tested.
Matching noisy area does not fix true area; escaping trajectories are excluded.


## Provenance
Inputs are small results files; no dumps are read.

In [1]:
import hashlib, json, subprocess, datetime
from pathlib import Path
SEED = 20261015
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Cargo.toml').exists())
EXP = ROOT / 'experiments/archive/2026-10-independent-area'
INPUTS = [EXP / 'results/analysis.json', ROOT / 'experiments/archive/2026-10-t1-excursion-tomography/results/t1b_cutoff_seeds.csv']
print('git', subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip())
print('UTC', datetime.datetime.now(datetime.timezone.utc).isoformat(), 'seed', SEED)
for p in INPUTS:
    print(p.relative_to(ROOT), hashlib.sha256(p.read_bytes()).hexdigest())
d = json.loads(INPUTS[0].read_text())
assert hashlib.sha256(INPUTS[1].read_bytes()).hexdigest() == d['input_sha256']
print('Producing commit', d['commit'], 'command', d['command'])


git da88e61b690507526ba4e2a2c017d9c7941beb91
UTC 2026-10-15T08:15:13.043584+00:00 seed 20261015
experiments/2026-10-15-independent-area/results/analysis.json 737ad21e7495302cfc70176a60b7adb4d6a86c3136db8bc521962fcaaeebf610
experiments/2026-10-14-t1-excursion-tomography/results/t1b_cutoff_seeds.csv e813c6f21f63421cb28e16df440f8317c0b3b715fa7718b74337edd9249e2b87
Producing commit da88e61b690507526ba4e2a2c017d9c7941beb91 command uv run experiments/2026-10-15-independent-area/scripts/analyse.py


## Data and method
Reuse k=22 and 25, excluding escapes. All six permutations assign separate groups
to area matching, kappa and cutoff. Average the six concordance fractions equally.
Bootstrap rules 2000 times with seed 20261016; never bootstrap pairs independently.
Heavy analysis: `scripts/analyse.py`. The bootstrap seed is analysis seed + 1.
## Results


In [2]:
print('Synthetic controls:', d['controls'])
for r in d['windows']:
    print('k', r['k'], 'tolerance', r['tolerance'], 'rules', r['n_rules'],
          'concordance', round(r['concordance'], 3),
          'CI', [round(x, 3) for x in r['bootstrap95']], r['verdict'])
    print('Role pair counts:', [x['pairs'] for x in r['roles']])


Synthetic controls: {'null': 0.49667267998075154, 'negative': 0.9571113303072005}
k 22 tolerance 0.05 rules 44 concordance 0.73 CI [0.605, 0.842] held
Role pair counts: [73, 73, 62, 62, 68, 68]
k 22 tolerance 0.1 rules 44 concordance 0.694 CI [0.573, 0.799] held
Role pair counts: [146, 146, 130, 130, 118, 118]
k 25 tolerance 0.05 rules 44 concordance 0.757 CI [0.617, 0.881] held
Role pair counts: [70, 70, 57, 57, 62, 62]
k 25 tolerance 0.1 rules 44 concordance 0.728 CI [0.603, 0.845] held
Role pair counts: [121, 121, 115, 115, 114, 114]


The registered concordance threshold and interval criterion hold at both windows.
This does not establish causality or constitute validation on new rules.
## Caveats and controls
Synthetic null and negative-association controls pass. They calibrate signs and
role handling, not scientific confounding. Area is noisy, rules are selected,
and escape exclusion can select outcomes. Windows share trajectories; intervals
are marginal. No rotor or random-walk growth estimator was changed.
## What this changes / next steps
The shared-trajectory area-matching confound cannot suffice to explain this signal.
The earlier seed-noise gate remains failed. Next obtain new trajectories matched
in size and design an intervention with power for the small residual effect.
## Open questions
Does the association persist at truly fixed area? Is the cutoff causal, or a
second response to an unmeasured property of the field?
